# 161 — Golden datasets, regresión y LLM-as-judge

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

Un **golden set** es un conjunto pequeño, curado y versionado de casos con salida esperada
verificada por humanos: cada ítem existe por una razón (caso borde, fallo histórico, riesgo). La
**evaluación de regresión** lo ejecuta ante cada cambio y lo accionable es el **diff de
veredictos** (qué pasaba y ahora falla), no la tasa global.

El **LLM-as-judge** (Zheng et al., arXiv:2306.05685) escala la calificación de respuestas
abiertas con >80 % de acuerdo con humanos en MT-Bench, pero con sesgos sistemáticos: **posición**
(favorece la primera respuesta en A/B), **verbosidad** (favorece lo largo), **autopreferencia**
(favorece su propia familia) y techo de capacidad (no detecta errores que él mismo cometería).


## 🧮 Mini-ejemplo: kappa de Cohen

Juez vs humano en 50 ítems: coinciden en 41 → acuerdo bruto p_o = 0.82. Pero por marginales
(juez acepta 70 %, humano 68 %) el azar ya produce p_e = 0.70·0.68 + 0.30·0.32 = 0.572.

```text
kappa = (0.82 - 0.572) / (1 - 0.572) = 0.579  → acuerdo MODERADO, no "casi perfecto"
```

Conclusión: el juez sirve para triaje, no como veredicto final sin auditoría humana.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("evaluation", seed=161)
show(result)


## Reflexión

1. ¿Por qué el diff de veredictos por ítem es más accionable que la tasa global de acierto en una
   suite de regresión? Da un caso donde el promedio sube y el despliegue debe bloquearse.
2. Tu LLM-judge prefiere la respuesta A el 78 % de las veces cuando A va primero, y el 60 % cuando
   va segundo. ¿Qué sesgo es y qué protocolo lo mitiga?
3. ¿Qué kappa mínimo exigirías para dejar que el juez apruebe respuestas sin revisión humana en un
   dominio médico, y por qué el umbral depende del costo del falso "aceptable"?
